In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv('IPL.csv')
df.head()

In [ ]:
df.columns = df.columns.str.strip()
df = df.dropna(subset=['runs_total'])



In [ ]:
df['venue'].unique()

In [ ]:
def analyze_venue(df, venue_name):

    venue_df = df[df['venue'] == venue_name]

    if venue_df.empty:
        print("Venue not found!")
        return

    print(f"\n Analysis for: {venue_name}")
    print("="*40)

    # total matches
    total_matches = venue_df['match_id'].nunique()
    print("Total Matches:", total_matches)

    # total scores per innings
    scores = venue_df.groupby(['match_id','innings'])['runs_total'].sum().reset_index()
    avg_score = scores['runs_total'].mean()
    print("Average Score:", round(avg_score,2))

    # powerplay runs per innings
    pp_runs = venue_df[venue_df['over'] <= 6] \
        .groupby(['match_id','innings'])['runs_total'].sum().reset_index()
    pp_avg = pp_runs['runs_total'].mean()

    # death runs per innings
    death_runs = venue_df[venue_df['over'] >= 16] \
        .groupby(['match_id','innings'])['runs_total'].sum().reset_index()
    death_avg = death_runs['runs_total'].mean()

    print("Avg Powerplay Runs:", round(pp_avg,2))
    print("Avg Death Runs:", round(death_avg,2))

    # chasing vs defending
    match_scores = scores.pivot_table(index='match_id', columns='innings', values='runs_total').reset_index()
    match_scores = match_scores[[col for col in match_scores.columns if col in ['match_id',1,2]]]
    match_scores.columns = ['match_id','inn1','inn2']

    match_scores['winner'] = np.where(match_scores['inn1'] > match_scores['inn2'], 1, 2)
    chasing_win_rate = (match_scores['winner'] == 2).mean()

    print("Chasing Win %:", round(chasing_win_rate*100,2))

    # insights
    print("\n Insights:")

    if avg_score > 170:
        print("- High scoring venue")
    elif avg_score > 150:
        print("- Moderate scoring venue")
    else:
        print("- Low scoring venue")

    if chasing_win_rate > 0.55:
        print("- Teams should prefer chasing")
    elif chasing_win_rate < 0.45:
        print("- Defending first is advantageous")
    else:
        print("- Toss has neutral impact")

    if pp_avg > death_avg:
        print("- Powerplay is more impactful")
    else:
        print("- Death overs are more impactful")

    # visualization
    import matplotlib.pyplot as plt
    import seaborn as sns

    plt.figure(figsize=(6,4))
    sns.histplot(scores['runs_total'], bins=20, kde=True)
    plt.title(f"Score Distribution - {venue_name}")
    plt.xlabel("Runs")
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
venues = sorted(df['venue'].dropna().unique())

for i, v in enumerate(venues):
    print(f"{i}: {v}")

choice = int(input("\nEnter venue number: "))
venue_name = venues[choice]

analyze_venue(df, venue_name)